In [1]:
import pandas as pd #自动填充0
csv_path = r'D:\workspace\Phase_diagram\GdCoAc_phase_diagram-main\experiment_results\batch3\run2.csv' 
output_path = r'experiment_results\batch3\run2_filled.csv'
df = pd.read_csv(csv_path, na_values=['', ' ', 'NaN', 'N/A', 'null', 'NULL'])
df_filled = df.fillna(0)
df_filled.to_csv(output_path, index=False)
print(f"新文件已保存: {output_path}")    

新文件已保存: experiment_results\batch3\run2_filled.csv


In [ ]:
# 初始布点
import numpy as np
import pandas as pd
from scipy.stats import qmc

def generate_experiment_data(N=100, seed=42):
    """Generate experimental data for experiment """
    D = 4  # Gd, Co, NaOH, NaHCO3
    np.random.seed(seed)

    # Define volume ranges (mL)
    n_Gd_range = [1.0, 3.0]
    n_Co_range = [0.5, 2.0]
    v_NaAc_range = [0.5, 1.0]
    v_NaOH_range = [2.0, 4.0]
    total_volume = 10.0  

    # Generate Latin Hypercube Samples
    sampler = qmc.LatinHypercube(d=D)
    portion = sampler.random(n=N)
    scaled_data = qmc.scale(
        portion,
        [n_Gd_range[0], n_Co_range[0], v_NaAc_range[0], v_NaOH_range[0]],
        [n_Gd_range[1], n_Co_range[1], v_NaAc_range[1], v_NaOH_range[1]]
    )

    # Create DataFrame
    df = pd.DataFrame(scaled_data, columns=["Gd(ClO4)3", "Co(Ac)2","NaHCO3", "NaOH"])

    # Calculate EtOH as the remaining volume to reach 10 mL
    df["EtOH"] = total_volume - df[["Gd(ClO4)3", "Co(Ac)2", "NaHCO3", "NaOH"]].sum(axis=1)

    # Drop rows where EtOH < 0
    df = df[df["EtOH"] >= 0].reset_index(drop=True)

    # Round the values
    df = df.round(2)
    df = df[["Gd(ClO4)3", "Co(Ac)2", "NaHCO3", "EtOH", "NaOH"]]
    # Save to CSV
    df.to_csv("Plan\\(Gd1-3 Co0.5-1 NaHCO3 OH2-4).csv", index=False)
    print("已保存")
    return df

df = generate_experiment_data(N=100, seed=42)

已保存


In [2]:
from data_split import process_folder
#数据处理
csv_path = r"D:\workspace\Phase_diagram\GdCoAc_phase_diagram-main\experiment_results"
try:
    df_stage1, df_stage2 = process_folder(csv_path, recursive=True)
    df_stage1.to_csv("training_data/for_model1.csv", index=False)
    df_stage2.to_csv("training_data/for_model2.csv", index=False)
    print(f"\n数据已保存为两个CSV文件")
except Exception as e:
    print(f"出错: {str(e)}")

处理文件: GdCoOAc_(OH2-3).csv
 样本数: 100，有晶体: 34，Gd42: 13
处理文件: GdCoOAc_(OH2.5-3.5).csv
 样本数: 100，有晶体: 40，Gd42: 17
处理文件: (Gd1-3 Co0.5-1 Na1-2 OH2-4).csv
 样本数: 100，有晶体: 45，Gd42: 28
处理文件: (Gd1-3 Co1-2 Na0.5-1 OH2-4).csv
 样本数: 100，有晶体: 54，Gd42: 45
处理文件: GdCoOAc_(OH3-4).csv
 样本数: 100，有晶体: 33，Gd42: 11
处理文件: run2.csv
 样本数: 100，有晶体: 34，Gd42: 15

文件夹处理完成！
共处理 6 个文件
合并后总样本数: 600
总有晶体样本: 240 (40.0%)
总Gd42晶体: 129 (53.8%)

数据已保存为两个CSV文件


In [3]:
from select_points import *
path1 = "training_data/for_model1.csv"
path2 = "training_data/for_model1.csv"
X1, Y1 = load_data(path1)
X2, Y2 = load_data(path2)
models1, behavior1 = train_models(X1, Y1)
models2, behavior2 = train_models(X2, Y2)

  0%|          | 0/32 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 168, number of negative: 252
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000539 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 380
[LightGBM] [Info] Number of data points in the train set: 420, number of used features: 4
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.400000 -> initscore=-0.405465
[LightGBM] [Info] Start training from score -0.405465
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


  0%|          | 0/32 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 168, number of negative: 252
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000046 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 389
[LightGBM] [Info] Number of data points in the train set: 420, number of used features: 4
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.400000 -> initscore=-0.405465
[LightGBM] [Info] Start training from score -0.405465
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

  0%|          | 0/32 [00:00<?, ?it/s]

...

[LightGBM] [Info] Number of positive: 168, number of negative: 252
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000043 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 381
[LightGBM] [Info] Number of data points in the train set: 420, number of used features: 4
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.400000 -> initscore=-0.405465
[LightGBM] [Info] Start training from score -0.405465
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

  0%|          | 0/32 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 168, number of negative: 252
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000036 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 384
[LightGBM] [Info] Number of data points in the train set: 420, number of used features: 4
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.400000 -> initscore=-0.405465
[LightGBM] [Info] Start training from score -0.405465
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

  0%|          | 0/32 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 168, number of negative: 252
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000036 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 380
[LightGBM] [Info] Number of data points in the train set: 420, number of used features: 4
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.400000 -> initscore=-0.405465
[LightGBM] [Info] Start training from score -0.405465
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

In [4]:
all_points = generate_lhs_samples(10000,random_state=42)
points_with_entropy1 = calculate_entropy(models1, behavior1, all_points)
points_with_entropy2 = calculate_entropy(models2, behavior2, all_points)

entropy1 = points_with_entropy1[:, 4]  # 第5列是熵
entropy2 = points_with_entropy2[:, 4]
point_entropy_avg = (entropy1 + entropy2)/2

points_with_all_entropy = np.hstack([
    all_points, 
    entropy1.reshape(-1, 1),
    entropy2.reshape(-1, 1),
    point_entropy_avg.reshape(-1, 1)
])
sorted_points = points_with_all_entropy[np.argsort(points_with_all_entropy[:, 6])[::-1]]
selected_points = select_points(sorted_points)
selected_array = np.array(selected_points)

experiment_df = pd.DataFrame(
            selected_array[:, :4].round(2),
            columns=['Gd(ClO4)3', 'Co(Ac)2', 'NaAc', 'NaOH']
        )
experiment_df['EtOH'] = (10 - selected_array[:, :4].sum(axis=1)).round(2)

experiment_df['entropy_model1'] = selected_array[:, 4].round(4)  # entropy1
experiment_df['entropy_model2'] = selected_array[:, 5].round(4)  # entropy2
experiment_df['avg_entropy'] = selected_array[:, 6].round(4)     # 平均熵

experiment_df.to_csv("Plan/run3.csv", index=False)